# Kubernetes networking deep-dive: service mesh traffic management, ingress controllers, and DNS resolution patterns

last_verified: 2026-10-12 · tool_version: n/a

## Purpose

This notebook maps the three layers that decide where traffic goes once it enters a cluster: the mesh layer that splits and shapes service-to-service traffic, the ingress layer that routes external requests to the right Service, and the DNS layer that turns short names into reachable addresses. Each section pairs a short reference explanation with a runnable model so the routing rule is visible in behavior, not just prose.


## When to use which layer

- **Service mesh traffic management** applies to east-west traffic: calls between services inside the cluster. Reach for it when a rollout needs weighted splits, mirrored shadow traffic, or uniform retry and timeout policy without editing every service.
- **Ingress controllers** apply to north-south traffic: requests arriving from outside the cluster. Reach for them when one entry point must fan out to many Services by host or path, and terminate encrypted connections at the edge.
- **DNS resolution patterns** apply everywhere names are used instead of addresses: Service discovery between pods, headless stateful endpoints, and external names. Reach for this section when a name resolves to the wrong address, or does not resolve at all.

Rule of thumb: a plain ClusterIP Service with cluster DNS is enough for stable internal callers; the mesh earns its place when traffic policy changes faster than service code, and ingress earns its place when external routing rules outgrow a single exposed Service.


## Prerequisites

- Access to a cluster with permission to list Services, Endpoints, and Ingress objects in at least one namespace.
- Familiarity with Deployments and Services; see `../docs/kubectl-imperative-vs-declarative.md` for the imperative versus declarative workflow contrast.
- A way to run the code cells below (a standard Python interpreter; every cell uses the standard library only).
- For the Verify checklist at the end, a shell with cluster access and a DNS lookup tool available inside a debug pod.


## Steps

1. Read Part 1 and run the traffic-split cell; confirm the observed shares match the declared weights.
2. Read Part 2 and run the routing cell; confirm every host/path case lands on the expected Service.
3. Read Part 3 and run the DNS cell; confirm short-name expansion and fully qualified parsing.
4. Work through the Verify checklist in a real cluster to confirm each modelled rule holds live.


## Part 1 — Service mesh traffic management

A service mesh inserts a proxy alongside each workload instance (the sidecar pattern) and configures the whole fleet of proxies from a central control plane. Application code keeps calling a plain Service name; the sidecar intercepts the call and applies the current routing policy before forwarding it.

The policies that matter most in practice:

- **Weighted splitting.** A share of requests goes to the new revision while the rest stays on the stable one. Weights shift gradually during a canary and snap back to zero on the first bad signal.
- **Mirroring.** A copy of live requests is sent to the new revision while responses still come from the stable one. Useful for validating behavior under real load without affecting callers.
- **Resilience policy in one place.** Retries with backoff, per-route timeouts, and circuit breaking are declared once and enforced by every proxy, instead of being reimplemented per service.
- **Identity-based rules.** Because each proxy presents a workload identity, routing and access decisions can key off who is calling, not just which address the packet came from.

Contrast with the default: without a mesh, a Service distributes connections across ready endpoints with simple balancing and no notion of revisions, so canary shares and shadow traffic have to be built into deploy tooling or application code. The mesh moves that logic into the network fabric, at the cost of running and configuring an extra proxy per workload. For help choosing between mesh implementations, see `../docs/choosing-service-mesh-istio-linkerd-consul.md`.


In [ ]:
import random

# last_verified: 2026-10-12 · kubernetes n/a
# Model of a mesh weighted split: 90% stable, 10% canary.
# The invariant the mesh enforces is that the weights always total 100;
# the simulation below shows what that split looks like over traffic.

split = {"stable": 90, "canary": 10}

total = sum(split.values())
assert total == 100, f"weights must total 100, got {total}"

rng = random.Random(42)  # fixed seed so the cell is reproducible
destinations = list(split)
weights = [split[d] for d in destinations]
observed = {d: 0 for d in destinations}
for _ in range(10_000):
    observed[rng.choices(destinations, weights=weights)[0]] += 1

for name, count in observed.items():
    print(f"{name}: {count / 100:.1f}% (configured {split[name]}%)")

# The canary share must stay within a small tolerance of the configured weight.
assert abs(observed["canary"] / 100 - split["canary"]) < 2, "split drifted from policy"
print("split matches policy")


## Part 2 — Ingress controllers

An Ingress resource declares how external requests map to Services: which hostnames are accepted, which path prefixes route where, and where encrypted connections terminate. An ingress controller is the process that watches those resources and programs the actual forwarding (reverse proxy, load balancer, or edge gateway) to match.

Key distinctions:

- **Resource versus controller.** The Ingress object is declarative intent; nothing happens until a controller that understands its class is running. The class field selects which controller owns a given resource when several are installed.
- **Host and path routing.** A single entry point can serve `api` traffic to one Service and `web` traffic to another, or split one hostname by path prefix. The longest matching path prefix wins for overlapping rules.
- **Edge termination.** Encrypted connections typically terminate at the ingress layer, with plain traffic forwarded inside the cluster. Certificate references live on the Ingress resource, not on each workload.
- **Exposure underneath.** The controller itself still needs a way to receive traffic, usually a LoadBalancer-type Service or a fixed node port, depending on the environment.

When routing needs outgrow one Ingress object (per-team ownership, distinct policy per route), the gateway-style APIs decompose the same idea into separate listener, route, and policy objects. The matching principle below stays the same regardless of which API shape is used.


In [ ]:
# last_verified: 2026-10-12 · kubernetes n/a
# Model of ingress host/path routing with longest-prefix matching.
# Rules are (host, path_prefix, service) triples in declaration order.

rules = [
    ("shop.example.internal", "/api/", "api-service"),
    ("shop.example.internal", "/", "web-service"),
    ("status.example.internal", "/", "status-service"),
]


def route(host, path):
    """Return the service for a request, or None when nothing matches."""
    best = None
    for rule_host, prefix, service in rules:
        if rule_host != host:
            continue
        if not path.startswith(prefix):
            continue
        if best is None or len(prefix) > len(best[1]):
            best = (rule_host, prefix, service)
    return best[2] if best else None


cases = [
    (("shop.example.internal", "/api/orders"), "api-service"),
    (("shop.example.internal", "/"), "web-service"),
    (("shop.example.internal", "/about"), "web-service"),
    (("status.example.internal", "/health"), "status-service"),
    (("unknown.example.internal", "/"), None),
]

for (host, path), expected in cases:
    got = route(host, path)
    status = "ok" if got == expected else f"MISMATCH (got {got})"
    print(f"{host}{path} -> {got} [{status}]")
    assert got == expected, f"routing error for {host}{path}"

print("all routes resolve as declared")


## Part 3 — DNS resolution patterns

Cluster DNS answers the question every pod asks constantly: what address does this name mean right now? The cluster runs a DNS service that watches Services and Endpoints and serves records derived from them, so service discovery works through ordinary name lookups with no client library.

The patterns to know:

- **Short names expand by namespace.** Inside a pod, `api` first resolves against the pod's own namespace, then against progressively shorter search suffixes. The fully qualified form is `api.<namespace>.svc.cluster.local`, and using it explicitly removes all ambiguity across namespaces.
- **Regular Services return one address.** A ClusterIP Service has a single stable virtual address, so DNS returns one record and balancing happens downstream at the connection layer.
- **Headless Services return the endpoints.** With no cluster address assigned, DNS answers with one record per ready pod address. Stateful workloads and custom client-side balancing rely on this.
- **External names alias out.** A Service of the external-name kind returns a canonical-name record pointing at a chosen outside hostname, letting in-cluster callers use a stable local name for an external dependency.
- **Search-list expansion has a cost.** Multi-label names are tried against each search suffix before the bare name is attempted, which multiplies upstream queries for names that only exist outside the cluster. Prefer fully qualified names (with a trailing dot for outside names) on hot paths.


In [ ]:
# last_verified: 2026-10-12 · kubernetes n/a
# Model of short-name expansion and fully qualified name parsing.
# Mirrors the search order a pod's resolver follows for a bare service name.

CLUSTER_DOMAIN = "cluster.local"


def expand(name, pod_namespace):
    """Expand a service name the way cluster DNS search suffixes would."""
    if name.endswith("."):
        return name  # already absolute; no search expansion applies
    if "." in name:
        return name  # multi-label names are tried as given first
    return f"{name}.{pod_namespace}.svc.{CLUSTER_DOMAIN}"


def parse(fqdn):
    """Split a fully qualified service name into (service, namespace)."""
    parts = fqdn.rstrip(".").split(".")
    assert len(parts) >= 5 and parts[2] == "svc", f"not a service FQDN: {fqdn}"
    return parts[0], parts[1]


assert expand("api", "orders") == "api.orders.svc.cluster.local"
assert expand("api", "billing") == "api.billing.svc.cluster.local"
assert expand("api.orders.svc.cluster.local", "billing") == "api.orders.svc.cluster.local"
assert expand("db.external.example.", "orders") == "db.external.example."
assert parse("api.orders.svc.cluster.local") == ("api", "orders")
assert parse("web.frontend.svc.cluster.local") == ("web", "frontend")

print(expand("api", "orders"))
print(parse("api.orders.svc.cluster.local"))
print("expansion and parsing behave as documented")


## Verify

Run this checklist in a real cluster to confirm each layer behaves as modelled above:

1. List the Services and their cluster addresses in a namespace, and confirm the stable name from the cells resolves to the listed address from a debug pod.
2. If a mesh is installed, inspect the current traffic weights for one canary-style route and confirm the shares add up to the whole, matching the invariant asserted in the split cell.
3. Send requests to each declared host and path from the routing cell through the ingress entry point, and confirm every request lands on the expected Service, including the longest-prefix overlap case.
4. Query a headless Service name from a debug pod and confirm the answer contains one record per ready endpoint rather than a single address.
5. Query the fully qualified form of a cross-namespace name and confirm it resolves without relying on search-suffix expansion.


## Common errors

| Symptom | Likely cause | Fix |
|---|---|---|
| Canary revision receives no traffic despite a weight change | Sidecar not injected on the new pods, or revision labels do not match the route selector | Confirm proxies run on the new pods and the labels match the declared subset |
| All traffic shifts at once instead of gradually | Weight update applied as an all-or-nothing replacement rather than a stepped change | Reapply the split in small steps and re-check the shares after each one |
| Ingress returns a not-found response for a declared path | Request reached a controller that does not own the resource class, or a path typo | Check the class field and compare the declared prefix with the requested path character by character |
| Host routes land on the wrong Service | Overlapping prefixes with the shorter rule shadowing the longer one in a hand-written config | Order rules so the longest prefix is evaluated first, as the routing cell does |
| Short name resolves in one namespace but fails in another | Bare name expands against the caller's namespace, which holds a different Service or none | Use the fully qualified `service.namespace` form for every cross-namespace call |
| Slow lookups for outside names | Multi-label outside names retried against every search suffix first | Use the absolute form with a trailing dot on hot paths |


## See also

- `../docs/choosing-service-mesh-istio-linkerd-consul.md` — chooser for mesh implementations, complementing the traffic-policy focus here.
- `../docs/kubernetes-helm-argocd-gitops-workflow.md` — how the routing declarations above travel from repository to cluster.
- `../docs/kubectl-imperative-vs-declarative.md` — imperative inspection commands useful for the Verify checklist.
